# From Vibecoding to Vericoding
## Interactive Demo: Session Types for AI Agent Communication

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafapra3008/cervellaswarm/blob/main/docs/blog/from-vibecoding-to-vericoding-demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/cervellaswarm-lingua-universale)](https://pypi.org/project/cervellaswarm-lingua-universale/)
[![Tests](https://img.shields.io/badge/tests-1820-brightgreen)](https://github.com/rafapra3008/cervellaswarm)

**Runtime-verified communication protocols for AI agents. Pure Python. Zero dependencies.**

This notebook demos [`cervellaswarm-lingua-universale`](https://pypi.org/project/cervellaswarm-lingua-universale/) in ~2 minutes. No signup. No GPU. Just run all cells.

---

## 0. Setup

In [ ]:
# Install (zero dependencies -- no runtime restart needed)
%pip install -q cervellaswarm-lingua-universale

import importlib.metadata
v = importlib.metadata.version("cervellaswarm-lingua-universale")
print(f"cervellaswarm-lingua-universale v{v} -- ready!")

## 1. The Problem: Untyped Agent Communication

Today, across AutoGen, CrewAI, LangGraph -- this is how AI agents talk:

In [ ]:
# How agents communicate in every major framework (2026)
message = {"role": "backend", "content": "here's the code", "type": "task_result"}

# Did the right agent send this? Nobody checks.
# Is it the right point in the conversation? Nobody knows.
# Is the content valid? We hope so.
print("Current state of the art:", message)
print()
print("No type checking. No protocol enforcement.")
print("The receiver guesses. Bugs hide until production.")

## 2. The Solution: Typed Protocols

Define who sends what, to whom, in what order. The type system enforces it.

In [ ]:
from cervellaswarm_lingua_universale import (
    Protocol, ProtocolStep, MessageKind,
)

# Define a code review protocol
code_review = Protocol(
    name="CodeReview",
    roles=("developer", "reviewer", "lead"),
    elements=(
        ProtocolStep(sender="developer", receiver="reviewer",
                     message_kind=MessageKind.TASK_RESULT,
                     description="Developer submits code for review"),
        ProtocolStep(sender="reviewer", receiver="lead",
                     message_kind=MessageKind.AUDIT_REQUEST,
                     description="Reviewer escalates for lead sign-off"),
        ProtocolStep(sender="lead", receiver="developer",
                     message_kind=MessageKind.AUDIT_VERDICT,
                     description="Lead delivers final verdict"),
    ),
)

print(f"Protocol : {code_review.name}")
print(f"Roles    : {', '.join(code_review.roles)}")
print(f"Steps    : {len(code_review.elements)}")
print()
for i, step in enumerate(code_review.elements, 1):
    print(f"  {i}. {step.sender} -> {step.receiver} : {step.message_kind.name}")

## 3. Runtime Violation Detection

If an agent sends the wrong message, to the wrong recipient, or out of order -- the checker catches it *immediately*.

In [ ]:
from cervellaswarm_lingua_universale import (
    SessionChecker, ProtocolViolation,
    TaskResult, TaskStatus,
)

# --- Happy path: follow the protocol ---
checker = SessionChecker(code_review, session_id="demo-001")

msg = TaskResult(
    task_id="T1", status=TaskStatus.OK,
    summary="Auth module refactored, 47 tests passing",
)
checker.send("developer", "reviewer", msg)
print("Step 1: developer -> reviewer : TaskResult  [OK]")
print(f"Protocol progress: step {checker.step_index}/{len(code_review.elements)}")

print()

# --- Violation: developer tries to skip reviewer and send to lead ---
checker2 = SessionChecker(code_review, session_id="demo-002")

try:
    checker2.send("developer", "lead", msg)  # Wrong! Protocol says developer -> reviewer first
except ProtocolViolation as exc:
    print("VIOLATION CAUGHT!")
    print(f"  Protocol : {exc.protocol}")
    print(f"  Session  : {exc.session_id}")
    print(f"  Expected : {exc.expected}")
    print(f"  Got      : {exc.got}")
    print()
    print("Not after deployment fails. Not after data corruption. Immediately.")

## 4. DSL Notation

Protocols can be written in a concise Scribble-inspired notation: `sender -> receiver : MessageKind;`

In [ ]:
from cervellaswarm_lingua_universale import render_protocol, parse_protocol

# Render our protocol to DSL
dsl_text = render_protocol(code_review)
print("=== Protocol as DSL ===")
print(dsl_text)

# Round-trip: parse it back
parsed = parse_protocol(dsl_text)
assert parsed.name == code_review.name
assert parsed.roles == code_review.roles
print("Round-trip verified: parse(render(P)).name == P.name")
print()

# You can also write protocols directly in the DSL
custom_dsl = """
protocol DelegateTask {
    roles regina, worker, guardiana;

    regina    -> worker    : TaskRequest;
    worker    -> regina    : TaskResult;
    regina    -> guardiana : AuditRequest;
    guardiana -> regina    : AuditVerdict;
}
"""
delegate = parse_protocol(custom_dsl)
print(f"Parsed from DSL: {delegate.name}, {len(delegate.elements)} steps")
for i, step in enumerate(delegate.elements, 1):
    print(f"  {i}. {step.sender} -> {step.receiver} : {step.message_kind.name}")

## 5. Elm/Rust-Style Error Messages

When something goes wrong, you get human-readable errors with suggestions -- in English, Italian, or Portuguese.

In [ ]:
from cervellaswarm_lingua_universale import (
    humanize, format_error, SUPPORTED_LOCALES,
)

print(f"Supported locales: {sorted(SUPPORTED_LOCALES)}")
print()

# Trigger a violation and get human-readable error
checker3 = SessionChecker(code_review, session_id="demo-003")

try:
    # Wrong sender: "lead" tries to start instead of "developer"
    checker3.send("lead", "reviewer", TaskResult(
        task_id="T1", status=TaskStatus.OK,
        summary="I went first, skipping the process!",
    ))
except ProtocolViolation as exc:
    # English -- concise
    err_en = humanize(exc, locale="en")
    print("=== English ===")
    print(format_error(err_en, verbose=False))
    print()

    # English -- verbose (with suggestions, like Elm/Rust)
    print("=== English (verbose, with suggestions) ===")
    print(format_error(err_en, verbose=True))
    print()

    # Italian
    err_it = humanize(exc, locale="it")
    print("=== Italiano ===")
    print(format_error(err_it, verbose=False))

## 6. Lean 4 Formal Verification

Generate Lean 4 theorem prover code from your protocol. Seven properties verified with mathematical proofs -- not tests.

In [ ]:
from cervellaswarm_lingua_universale import (
    generate_lean4, lean4_available, FLAT_PROPERTIES,
)

print(f"Lean 4 installed locally: {lean4_available()}")
print(f"Properties we can prove: {len(FLAT_PROPERTIES)}")
for prop in sorted(FLAT_PROPERTIES, key=lambda p: p.value):
    print(f"  - {prop.value}")
print()

# Generate Lean 4 code for our CodeReview protocol
lean_code = generate_lean4(code_review)
print(f"Generated {len(lean_code.splitlines())} lines of Lean 4:")
print()
print(lean_code)
print()
print("Each 'by decide' is a mathematical proof, not a test.")
print("Run 'lean --json' locally to machine-check every theorem.")

## 7. Static Property Verification

Check structural properties of your protocol *before* any agent runs.

In [ ]:
from cervellaswarm_lingua_universale import (
    parse_spec, check_properties, PropertyVerdict,
)

spec = parse_spec("""
properties for CodeReview:
    always terminates
    no deadlock
    all roles participate
""")

report = check_properties(code_review, spec)

print("=== Static Verification Report ===")
for result in report.results:
    verdict = "PROVED" if result.verdict == PropertyVerdict.PROVED else (
        "VIOLATED" if result.verdict == PropertyVerdict.VIOLATED else "UNKNOWN"
    )
    symbol = {"PROVED": "+", "VIOLATED": "!", "UNKNOWN": "?"}[verdict]
    print(f"  [{symbol}] {result.spec.kind.value:24s} {verdict}")
    if result.evidence:
        print(f"      Evidence: {result.evidence}")
print()
print(f"Result: {report.passed_count}/{len(report.results)} properties proved")
print(f"All passed: {report.all_passed}")

## 8. Confidence as a First-Class Type

Not `{"confidence": "high"}` (a string). A real type that composes, propagates, and means something.

In [ ]:
from cervellaswarm_lingua_universale import (
    ConfidenceScore, ConfidenceSource,
    Confident, CompositionStrategy, compose_scores,
)

# Today: a string nobody checks
print("Today:", {"output": "bug found", "confidence": "high"})
print()

# With Lingua Universale: typed confidence
audit  = ConfidenceScore(value=0.95, source=ConfidenceSource.AUDIT)
self_r = ConfidenceScore(value=0.80, source=ConfidenceSource.SELF_REPORTED)

# Compose conservatively (take the minimum)
composed = compose_scores((audit, self_r), strategy=CompositionStrategy.MIN)
print(f"Audit confidence     : {audit.value:.2f} ({audit.source.value})")
print(f"Self-reported conf.  : {self_r.value:.2f} ({self_r.source.value})")
print(f"Composed (MIN)       : {composed.value:.2f} ({composed.source.value})")
print()

# Confident[T] wraps any value with its confidence
result = Confident(value="Auth module has a timing vulnerability", confidence=composed)
print(f"Confident[str].value   : {result.value!r}")
print(f"Confident[str].is_high : {result.is_high}")
print()

# .map() preserves confidence through transformations
upper = result.map(str.upper)
print(f"After .map(str.upper)  : {upper.value!r}")
print(f"Confidence preserved   : {upper.confidence.value}")

## 9. Trust Composition

A 4-tier trust model where delegation *always attenuates* -- a chain can never trust more than its weakest link.

In [ ]:
from cervellaswarm_lingua_universale import (
    TrustScore, TrustTier, AgentRole,
    trust_tier_for_role, compose_chain, chain_confidence,
)

# Each role has a default trust tier
for role in [AgentRole.REGINA, AgentRole.GUARDIANA_QUALITA, AgentRole.BACKEND]:
    tier = trust_tier_for_role(role)
    print(f"  {role.value:<28} tier={tier.value}")
print()

# Trust chain: Regina (verified) -> Backend (standard) -> External (untrusted)
chain = compose_chain((
    TrustScore(value=1.0,  tier=TrustTier.VERIFIED,  reason="regina"),
    TrustScore(value=0.75, tier=TrustTier.STANDARD,  reason="backend"),
    TrustScore(value=0.5,  tier=TrustTier.UNTRUSTED, reason="external"),
))

print(f"Chain: VERIFIED(1.0) x STANDARD(0.75) x UNTRUSTED(0.5)")
print(f"Result: {chain.value:.4f}  tier={chain.tier.value}")
print()
print("Privilege Attenuation: the chain can NEVER exceed 0.375")
print("This is a mathematical guarantee, not a convention.")
print()

# Combine trust chain with output confidence
final = chain_confidence(
    trust_chain=(TrustScore(value=1.0, tier=TrustTier.VERIFIED),
                 TrustScore(value=0.75, tier=TrustTier.STANDARD)),
    output_confidence=ConfidenceScore(value=0.9, source=ConfidenceSource.AUDIT),
)
print(f"Trust(1.0 x 0.75) x Confidence(0.9) = {final.value:.4f}")
print("Real confidence = trust_in_chain * output_confidence")

## 10. Observable Sessions

Attach a monitor to any session. Every message, every branch, every violation -- captured as structured events.

In [ ]:
from cervellaswarm_lingua_universale import (
    ProtocolMonitor, EventCollector, SessionChecker,
    DelegateTask,  # pre-built standard protocol
    TaskRequest, TaskResult, TaskStatus,
    AuditRequest, AuditVerdict, AuditVerdictType,
)

print(f"Built-in protocol: {DelegateTask.name}")
print(f"  Roles: {', '.join(DelegateTask.roles)}")
print(f"  Steps: {len(DelegateTask.elements)}")
print()

# Attach monitor to capture every event
monitor = ProtocolMonitor()
collector = EventCollector()
monitor.add_listener(collector)

checker = SessionChecker(DelegateTask, session_id="monitored-001", monitor=monitor)

In [ ]:
# Run the full 4-step protocol
steps = [
    ("regina", "worker", TaskRequest(task_id="T1",
        description="Implement OAuth2 login", target_files=("auth.py",))),
    ("worker", "regina", TaskResult(task_id="T1",
        status=TaskStatus.OK, summary="OAuth2 implemented, 23 tests pass")),
    ("regina", "guardiana", AuditRequest(audit_id="A1",
        target="T1", checklist=("security", "tests", "quality"))),
    ("guardiana", "regina", AuditVerdict(audit_id="A1",
        verdict=AuditVerdictType.APPROVED, score=9.5,
        checked=("security", "tests", "quality"))),
]

for sender, receiver, msg in steps:
    checker.send(sender, receiver, msg)
    print(f"  {sender:10} -> {receiver:10} : {type(msg).__name__:16} [OK]")

print()
print(f"Session complete : {checker.is_complete}")
print(f"Events captured  : {len(collector.events)}")
for evt in collector.events:
    print(f"  {type(evt).__name__}")

## What You Just Saw

| Feature | What it replaces |
|---------|------------------|
| Typed protocols | Hoping agents agree on message order |
| Runtime `SessionChecker` | Silent failures, corrupted state |
| DSL notation | Ad-hoc JSON schemas |
| Elm/Rust-style errors | Stack traces nobody reads |
| Lean 4 generation | "Trust me, it works" |
| Static spec checking | Manual protocol review |
| `Confident[T]` | `{"confidence": "high"}` strings |
| Trust chains | Untracked delegation |
| Protocol monitor | Scattered logs |

All of this in **pure Python. Zero dependencies. 1,820 tests. 98% coverage. 13 modules.**

---

## Next Steps

**Liked what you saw?**

- **Install:** `pip install cervellaswarm-lingua-universale`
- **GitHub:** [github.com/rafapra3008/cervellaswarm](https://github.com/rafapra3008/cervellaswarm) -- 9 packages, 3,791 tests, Apache 2.0
- **PyPI:** [pypi.org/project/cervellaswarm-lingua-universale](https://pypi.org/project/cervellaswarm-lingua-universale/)
- **Blog post:** [From Vibecoding to Vericoding](https://github.com/rafapra3008/cervellaswarm/blob/main/docs/blog/from-vibecoding-to-vericoding.md) -- the full story with 74 research sources

**Questions? Found a bug?** Open an issue on GitHub. We respond fast.

**What's next:** Fase B (confidence, trust, codegen, intent, spec, errors) is complete. Fase C -- CervellaLang, a programming language where protocols are native constructs -- starts 2027. Follow the repo.

---

*CervellaSwarm is built by Rafa and Cervella -- a human-AI partnership since December 2025. Apache 2.0.*

*"Ultrapassar os proprios limites" -- to surpass one's own limits.*